# Week 6: The Finite Difference Method & Quantum Mechanics

## Library Imports Go Here

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import time
import scipy.sparse as sparse
import scipy.sparse.linalg as linalg

## Generic Finite Difference Code


    
The essence of the finite difference method is that it allows us to replace an expression involving spatial derivatives such as

$$
A(x)\psi'' + B(x)\psi' + C(x)\psi
$$

subject to the boundary conditions

$$
\psi(x_{\mathrm{left}}) = \psi(x_{\mathrm{right}}) = 0
$$

with a linear algebra expression in which "$\psi(x)$" becomes a vector $\vec{\psi}$ of $N$ components, and the full expression is represented as

$$
\mathbb{M}\vec{\psi}
$$

where $\mathbb{M}$ is a $N \times N$ matrix.  This expression then appears in various different types of differential equations, which we can then use linear algebra to solve.

The most challenging aspect of this is generating the matrix $\mathbb{M}$.

### A Spatial Grid Function


    
Write a (simple) function that takes as inputs the numerical parameters $x_{\mathrm{left}}$, $x_{\mathrm{right}}$, and $N$ and outputs an array of positions $\{x_1, x_2, ..., x_N\}$ for use in the finite difference method.  Remember that the grid does *not* include the boundary values $x_{\mathrm{left}}$ and $x_{\mathrm{right}}$ themselves.  Test your code using the choices

$$
x_{\mathrm{left}} = 1, \hspace{.5in} x_{\mathrm{right}} = 2, \hspace{.5in} N = 4
$$

and as a **discussion**, confirm that the output is correct.

In [3]:
def spatial(x_left, x_right, N):
    return np.linspace(x_left, x_right, N+2)[1:-1]

spatial(1,2,4)

array([1.2, 1.4, 1.6, 1.8])

### A Projection Function



Now write a function that takes as inputs a function f(x) and a spatial grid (that is, a numpy array) $\{x_1, x_2, \dots, x_N\}$, and outputs a numpy array of that function projected onto the grid: $\{f(x_1), f(x_2), \dots, f(x_N)\}$.  Test your code using the same grid as above, and with the two functions

$$
f_{(1)}(x) = x^2 \,, \hspace{1in} 
f_{(2)}(x) =
\begin{cases}
  0 & \text{for } x < 1.5 \\
  1 & \text{for } x \ge 1.5
\end{cases}
$$


As a **discussion** confirm that your outputs are correct.

In [4]:
def proj(f, grid):
    array = []
    for i in range(len(grid)):
        array.append(f(grid[i]))
    return array

def f(x):
    return float(x**2)

grid = spatial(1,2,4)
proj(f, grid)

[1.44, 1.9599999999999997, 2.5600000000000005, 3.24]

### A Matrix Generating Function

#### Creating the Matrix



Construct a function that takes as inputs the three functions $A(x)$, $B(x)$, and $C(x)$, along with the spatial grid array, and outputs the matrix $\mathbb{M}$ (as an array.)  

Hints:  

1)  The positions "x" at which you need to evaluate the functions A, B, and C, are the values in the input array, with the position in the array corresponding to the row of the matrix.

2)  You can extract the values $\Delta$ and $N$ from the grid itself.

3)  You may (or may not) find it useful to use your projection function here.

In [5]:
def matrix(A, B, C, grid):
    del_x = grid[1] - grid[0]
    x_int = grid
    N = len(x_int)

    a = proj(A, x_int) 
    b = proj(B, x_int)  
    c = proj(C, x_int)  

    M = np.zeros((N, N))

    for i in range(N):
        M[i, i] = c[i] - ((2 * a[i]) / (del_x**2))

    for i in range(1, N):
        M[i, i-1] = (a[i] / (del_x**2)) - (b[i] / (2 * del_x))
    for i in range(0, N-1):
        M[i, i+1] = (a[i] / (del_x**2)) + (b[i] / (2 * del_x))

    return M

#### Testing for Predictable Results



Consider the functions

$$
A(x) = 2x \, , \hspace{.5in} B(x) = x^2 \, , \hspace{.5in} C(x) = 1
$$

and the grid with $x_{\mathrm{left}} = 1$, $x_{\mathrm{right}} = 2$, and $N = 4$.

As a **discussion**, work out what each component of the matrix should be.  Then, confirm your results are correct.  (Yes, this is a bit tedious.  But it is very common to get this code wrong in small, subtle ways, and this test is designed to protect against that).

In [6]:
def A(x):
    return 2*x
def B(x):
    return x**2
def C(x):
    return 1

grid = spatial(1, 2, 4)
matrix(A, B, C, grid)

array([[-119. ,   63.6,    0. ,    0. ],
       [  65.1, -139. ,   74.9,    0. ],
       [   0. ,   73.6, -159. ,   86.4],
       [   0. ,    0. ,   81.9, -179. ]])

### A Sparse Matrix Generating Function



You will notice that the matrix we are generating has a lot of zeroes in it, particularly when the value $N$ is large.  Creating and storing a matrix with a lot of zero values wastes memory in the computer, and requires more time to do anything with, than it really should.

A better approach is to create and store the matrix we are building as a type of "sparse array" in the `scipy.sparse` library known as a `coo_matrix`, which keeps track only of the entries that are non-zero, and the locations of those entries.  For example, the matrix

$$
\left[\begin{array}{cccc} 0 & 0 & 0 & 0 \\ 0 & 0 & 87 & 0 \\ 0 & 0 & 0 & 0 \\ 0 & 23 & 0 & 0 \end{array}\right]
$$

is stored simply as

$$
\begin{array}{ccc}
(1, 2) & \hspace{.5in} & 87 \\
(3, 1) & \hspace{.5in} & 23
\end{array}
$$

(The first non-zero element of the array is at row 1 and column 2, and has value 87.  The second non-zero element of the array is at row 3 and column 1, and has value 23).  All told, the computer needs to store 6 numbers using this construction, as opposed to the 16 it stores if we build it as a normal array.

To create a coo_matrix, we can use the command

`coo_matrix((data, (rows, columns)), shape = (N, N)`

where `data`, `rows`, and `columns` are each arrays of values.  For example, the above matrix has:

`data = np.array([87, 23])`

`rows = np.array([1, 3])`

`columns = np.array([2, 1])`

Your next job is to write a function that produces a sparse array version of the same matrix as before, but we will break it into steps to make the task more manageable.

In [7]:
def sparseMatrix(A, B, C, grid):
    del_x = grid[1] - grid[0]
    N = len(grid)

    diag_row  = np.array([i for i in range(N)])
    diag_col  = np.array([i for i in range(N)])
    diag_data = np.array([C(grid[i]) - (2 * A(grid[i])) / (del_x**2) for i in range(N)])

    up_row  = np.array([i for i in range(0, N-1)])
    up_col  = np.array([i+1 for i in range(0, N-1)])
    up_data = np.array([(A(grid[i]) / (del_x**2)) + (B(grid[i]) / (2 * del_x)) for i in range(0, N-1)])

    down_row  = np.array([i for i in range(1, N)])
    down_col  = np.array([i-1 for i in range(1, N)])
    down_data = np.array([(A(grid[i]) / (del_x**2)) - (B(grid[i]) / (2 * del_x)) for i in range(1, N)])

    rows = np.concatenate((diag_row, up_row,  down_row))
    cols = np.concatenate((diag_col, up_col,  down_col))
    data = np.concatenate((diag_data, up_data, down_data))

    return sparse.coo_matrix((data, (rows, cols)), shape=(N, N))

#### A Very Simple Test


    
Begin by testing out this new object by constructing the sparse matrix described above -- make arrays of the rows, columns, and data values, and use those as inputs to the `coo_matrix` command.  To check that your results are correct:

1)  Print out the sparse matrix you have constructed.

2)  Convert that matrix into a normal array using the command `.toarray()`, and print that out as well.

In [8]:
#1
print(sparseMatrix(A, B, C, grid))
print(sparseMatrix(A, B, C, grid).toarray())

<COOrdinate sparse matrix of dtype 'float64'
	with 10 stored elements and shape (4, 4)>
  Coords	Values
  (0, 0)	-119.00000000000006
  (1, 1)	-139.00000000000006
  (2, 2)	-159.00000000000009
  (3, 3)	-179.00000000000009
  (0, 1)	63.60000000000003
  (1, 2)	74.90000000000003
  (2, 3)	86.40000000000005
  (1, 0)	65.10000000000002
  (2, 1)	73.60000000000004
  (3, 2)	81.90000000000003
[[-119.    63.6    0.     0. ]
 [  65.1 -139.    74.9    0. ]
 [   0.    73.6 -159.    86.4]
 [   0.     0.    81.9 -179. ]]


#### Diagonal Elements

<font color = blue>
    
Now write a function whose outputs will be just the three arrays of rows, columns, and data values necessary to create just the *diagonal* elements of the matrix $\mathbb{M}$ we are interested in.  

It should take as inputs the functions $A(x)$ and $C(x)$, along with the grid itself.  You may (or may not) find it useful to use your projection function here.

Test it out using the same grid and functions you used to test your first matrix making function.  Take the outputs of the function and use them to create a sparse-matrix, and confirm the values are the same as in the previous function.

**Note**:  The point of using the sparse matrices is to increase efficiency -- so make sure you're not using "append" to build your arrays here!

#### Off-Diagonals

<font color = blue>
    
Now create two more functions to produce the upper and lower off-diagonals.  Again, each function should output the arrays of rows, columns, and data values necessary as inputs to the function `coo_matrix`.  These functions should have as inputs the functions A(x) and B(x), along with the spatial grid.

Test your functions again using the choices of A, B, and C and the spatial grid used earlier, and again use the results to create sparse matrices and compare them with your previous matrix generating code.

#### The Sparse Matrix Maker

<font color = blue>
    
Now create the full code to produce your sparse matrix.

1)  This function should take as inputs the functions A, B, C and the spatial grid

2)  It should *call* the functions you have just written to produce arrays of rows, columns, and data corresponding to different parts of the full matrix

3)  You can then use the `numpy` command `concatenate` to (efficiently) combine multiple arrays into a single array.

4)  The function should output the matrix as a `coo_matrix`.

Once again, test your function using the same A, B, and C functions and the same spatial grid.

### Comparing Efficiency

<font color = blue>

Now we want to compare the efficiency of working with these two types of matrices.  

#### An Analytic Argument

<font color = blue>

As a **discussion**, compute the total number of values stored to make an $N \times N$ matrix $\mathbb{M}$ using each of the two methods.  Based on this, what do you expect to see when you look at timing behaviors?

For the normal method, you store exactly $N^2$ numbers.

For the sparse array method, you store 3 numbers for each non-zero element of the matrix.  The total number of non-zero elements is $3N - 2$, so the total number of numbers stored is $9N - 6$.

We can see that as $N$ becomes large, the time required to create the normal matrix should grow quadratically, while the time required to create a sparse matrix should grow linearly.  This means that for large matrices the sparse matrix should be much more efficient -- though for smaller matrices it won't necessarily be.

#### Timing Functions

<font color = blue>

Write a function that takes as input the value $N$, and returns the amount of time required to build the matrix $\mathbb{M}$ using your original matrix generating function.

Then, do the same thing for your sparse matrix generating function.

#### Timing Data

<font color = blue>

Create an array of values of $N$ between 10 and 1000 in steps of 10.  Then, create associated timing arrays for the two different matrix makers.

#### Graphical Comparison

<font color = blue>

Create a graph of your timing data so that you can compare the two methods.

## Energy Eigenstates and Eigenvalues in Quantum Mechanics

<font color = blue>
    
Our first use for the finite difference method will be in solving for states of definite energy (and the energies themselves) in 1D quantum mechanics.  In this case, our specific math problem is written (in ND form) as

$$
-\frac{1}{2}\frac{d^2\psi}{d\tilde{x}^2} + \tilde{V}(\tilde{x})\psi = \tilde{E}\psi
$$
where we assume the wavefunction satisfies boundary conditions
$$
\psi(\pm\infty) = 0
$$
and we want to find both the energies $\tilde{E}$ and the associated states $\psi$ that solve this problem.


Once we have used the finite difference method to convert this into a linear algebra problem, it reads
$$
\mathbb{H}\vec{\psi} = \tilde{E}\vec{\psi}
$$

and we want to find the eigenvalues $\tilde{E}_{(k)}$ and eigenvectors $\vec{\psi}_{(k)}$ that solve the problem.  To do this, we will be using the function `scipy.sparse.linalg.eigsh`, intsead of writing our own code.  (Notice that this function will work directly with matrices in "sprarse" form).

### The Hamilonian Matrix

<font color = blue>

First we want to write the code that generates the matrix $\mathbb{H}$.  This code should take as inputs just the potential function $V(\tilde{x})$, and the numerical grid parameters $\tilde{x}_{\infty}$ and $N$, and then *call the functions you wrote in the previous section* to construct and output the matrix as a sparse matrix.

Note that in this case, our grid endpoints will be $-x_{\infty}$ and $+x_{\infty}$ (they both use the same numerical parameter).

### The Simple Harmonic Oscillator

<font color = blue>
    
Now we're going to specialize to the case

$$
V(\tilde{x}) = \frac{1}{2}\tilde{x}^2
$$

which is the (non-dimensionalized) quantum simple harmonic oscillator.  In this case, the problem of finding the states of definite energy and their energies has an analytic solution.  The energy spectrum is given by

$$
\tilde{E}_{(n)} = \frac{1}{2} + n \, , \hspace{.5in} n = 0, 1, 2, 3, \dots \, .
$$

The lowest three corresponding states are

$$
\psi_{(0)}(\tilde{x}) = \frac{1}{\pi^{1/4}} \, e^{-\tilde{x}^2/2}
\hspace{.75in}
\psi_{(1)}(\tilde{x}) = \frac{\sqrt{2}}{\pi^{1/4}} \, \tilde{x} \, e^{-\tilde{x}^2/2}
\hspace{.75in}
\psi_{2}(\tilde{x}) = \frac{\sqrt{2}}{\pi^{1/4}} \, \left(\tilde{x}^2 - \frac{1}{2}\right) \, e^{-\tilde{x}^2/2}
$$

#### The Eigenvalues

<font color = blue>

Use your code to generate the Hamiltonian matrix for the QSHO problem, with grid parameters $x_{\infty} = 100$ and $N = 100000$. Then, use the function `eigsh` from `scipy.sparse.linalg` to find the lowest 10 eigenvalues.  

Create a plot that shows those eigenvalues vs. the quantum number $n$, and on the same graph plot the theoretical values.  Confirm that they agree.

**Note** read the documentation for this function carefully: notice that the number of eigenvalues you get out is an input parameter, as is a parameter called "sigma", which determines whether the eigenvalues you get start with the smallest value, or with the largest.  Notice also that the *output* includes both the eigenvalues, and the eigenvectors.

#### The Low-Lying States

<font color = blue>

Now, examine the eigenstates.  Create three graphs showing the lowest three numerical eigenstates, together with the analytic predictions for them.  Note that:

1)  The eigenvectors that come out of `eigsh` are organized into a matrix where each *column* of the matrix is a different vector (not each row).

2)  The eigenvectors have components $\psi_i = \psi(x_i)$ where $x_i$ are the positions on the spatial grid.  

3)  The eigenvectors are normalized "as vectors", which follows a different rule than the wavefunction states do -- to correct for this you will need to multiply each vector by an additional constant that you can tune by hand (and that constant may be negative!)

#### The Grid Parameter $\Delta$

<font color = blue>

Now we want to examine the significance of the numerical parameters, starting with $\Delta$.  Using the $n = 9$ state of the QSHO, find and plot this state for four different choices of $N$, while leaving $x_{\infty} = 100$:

$N = 200, 400, 800, 1600, 3200$

As a **discussion** compute $\Delta$ for each of these cases, and examine the effect of gradually increasing $\Delta$ on the method: what needs to be true in order for the method to "do a good job" of approximating the state?

#### The Grid Parameter $x_{\infty}$

<font color = blue>

Now we want to examine the effect of changing $x_{\infty}$, while leaving $\Delta = 0.01$ fixed.    Create a graph that shows the $n = 9$ state with $x_{\infty} = \{100, 50, 10, 5, 1\}$.  For each case, you will also need to work out the value of $N$ necessary to leave $\Delta = 0.01$.

Then **discuss** the results.  What needs to be true of $x_{\infty}$ in order for the method to "do a good job" on the state?

### Comparing Potentials*

<font color = blue>

Now we want to look at how changing the potential energy function (the physical system our particles are in) affects the energies and states.  We will consider two other potentials (neither of which can be approached analytically), to compare with the SHO potential:

$$
\tilde{V}_{SHO}(\tilde{x}) = \frac{1}{2}\tilde{x}^2 \, , \hspace{.5in} \tilde{V}_{abs}(\tilde{x}) = |\tilde{x}| \, , \hspace{.5in} \tilde{V}_{quad}(\tilde{x}) = \frac{1}{4} \tilde{x}^4
$$

In your discussions, think about what is *the same*, and what is *different* as you change the potentials.  Also think about what the behavior of the potentials would imply classically, and how that is (or isn't) reflected in the quantum mechanical results.

#### The Potentials Themselves

<font color = blue>

Begin by simply plotting these three potentials together, so that you can see the ways in which they are similar, and the ways in which they are different.

#### The Energy Spectra

<font color = blue>

Now find the lowest ten energy eigenvalues for each of the three potentials, and create a plot that shows all three energy spectra together.  **Discuss** the physics of what you see thoroughly (and keep in mind the differences in the potentials themselves as you do!)  Use the grid parameters $x_{\infty} = 100$ and $N = 100000$.

#### The Eigenstate Wavefunctions

<font color = blue>

Now we want to explore the wavefunctions themselves.  Create a graph that shows the ground state wavefunction for each of the three potentials.  Then do the same for the $n = 1$ and $n = 2$ states.  Again, **discuss** the physics of the results thoroughly.